# Ingesta de relaciones película-idioma

Lee la carpeta `movie_language` (JSON multilínea) desde Bronze, aplica un esquema explícito, estandariza las columnas y guarda el resultado como Parquet en `silver/movies_languages` (modo `overwrite`).

### Configuración

Carga las rutas de almacenamiento y las funciones comunes, y lee el parámetro `p_environment`.

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

### Lectura del archivo

Primero lee la carpeta `movie_language` con inferencia para inspeccionarlo y luego lo relee con un esquema explícito.

In [0]:
movie_language_df = (
    spark.read
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/movie_language")
    )
display(movie_language_df)

In [0]:
movie_language_schema = StructType([
    StructField("movieId", StringType()),
    StructField("languageId", IntegerType()),
    StructField("languageRoleId", IntegerType())
])

movie_language_df = (
    spark.read
        .schema(movie_language_schema)
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/movie_language")
)
display(movie_language_df)

### Transformación de columnas

Renombra `movieId` y `languageId`, descarta `languageRoleId` y agrega `ingestion_date` y `env`.

In [0]:
movie_language_df = (
    movie_language_df
        .withColumnsRenamed(
            {
                "movieId": "movie_Id",
                "languageId": "language_id"
            }

        )
        .withColumns(
            {
                "env": lit(v_env)
            }
        )
).drop("languageRoleId")

movie_language_df = add_ingestion_date(movie_language_df)

display(movie_language_df)

### Escritura en Silver

Guarda el resultado como Parquet en `silver/movies_languages` en modo `overwrite`.

In [0]:
(
    movie_language_df
        .write
        .format("parquet")
        .mode("overwrite")
        .save(f"{silver_folder_path}/movies_languages")
)

### Fin

Devuelve `Success` al notebook orquestador.

In [0]:
dbutils.notebook.exit("Success")